# Chunks

In [33]:
import json
import re
import numpy as np
import pandas as pd
from collections import Counter

### 데이터 로드

In [34]:
BASE_PATH = "/content"
with open(f"{BASE_PATH}/ingredient_merged.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"총 행 수: {len(data)}")
print(f"컬럼 목록: {list(data[0].keys())}")

총 행 수: 87712
컬럼 목록: ['ingredient_ko', 'ingredient_en', 'pc_rating', 'pc_effect', 'pc_category', 'pc_description', 'coos_function', 'coos_type', 'coos_cas_no', 'coos_country', 'coos_kr_restricted', 'coos_cn_restricted', 'coos_tw_restricted', 'coos_jp_restricted', 'coos_eu_restricted', 'coos_asean_restricted', 'coos_ai_description', 'coos_score', 'coos_data_grade', 'hw_product_id', 'hw_product_name', 'hw_brand_name', 'hw_avg_ratings', 'hw_review_count', 'hw_price', 'hw_consumer_price', 'hw_primary_attr', 'hw_topics_positive', 'hw_topics_negative', 'hw_ingredient_count', 'hw_ingredient_id', 'hw_ewg', 'hw_ewg_data_availability_text', 'hw_is_allergy', 'hw_purpose', 'hw_purposes', 'hw_limitation', 'hw_forbidden', 'hw_category']


### 청크 가중치 설정

In [35]:
CHUNK_WEIGHT = {
    "ewg":        1.0,
    "basic_info": 1.0,
    "expert":     1.0,
    "review":     1.0,
}

print(f"청크 가중치: {CHUNK_WEIGHT}")

청크 가중치: {'ewg': 1.0, 'basic_info': 1.0, 'expert': 1.0, 'review': 1.0}


### ❕❕❕EWG 점수 정규화 함수 (추가)

In [36]:
# coos_score unique 값 확인
coos_scores = set(row.get("coos_score") for row in data if row.get("coos_score"))
print(f"coos_score unique 값 ({len(coos_scores)}개):")
print(sorted(coos_scores)[:50])  # 너무 많으면 50개만

print()

# hw_ewg unique 값 확인
hw_ewgs = set(row.get("hw_ewg") for row in data if row.get("hw_ewg"))
print(f"hw_ewg unique 값 ({len(hw_ewgs)}개):")
print(sorted(hw_ewgs))

coos_score unique 값 (36개):
['0', '[안전] 1 등급', '[안전] 1-2 등급', '[안전] 2 등급', '[위험] 1-9 등급', '[위험] 10 등급', '[위험] 2-7 등급', '[위험] 2-9 등급', '[위험] 3-7 등급', '[위험] 4-7 등급', '[위험] 5-8 등급', '[위험] 5-9 등급', '[위험] 6-9 등급', '[위험] 7 등급', '[위험] 7-10 등급', '[위험] 8 등급', '[위험] 8-10 등급', '[위험] 9 등급', '[주의] 1-3 등급', '[주의] 1-4 등급', '[주의] 1-5 등급', '[주의] 1-6 등급', '[주의] 2-3 등급', '[주의] 2-4 등급', '[주의] 2-5 등급', '[주의] 2-6 등급', '[주의] 3 등급', '[주의] 3-4 등급', '[주의] 3-5 등급', '[주의] 3-6 등급', '[주의] 4 등급', '[주의] 4-5 등급', '[주의] 4-6 등급', '[주의] 5 등급', '[주의] 5-6 등급', '[주의] 6 등급']

hw_ewg unique 값 (34개):
['1', '10', '1_2', '1_3', '1_4', '1_5', '1_6', '1_9', '2', '2_3', '2_4', '2_5', '2_6', '2_7', '3', '3_10', '3_4', '3_5', '3_6', '3_7', '3_9', '4', '4_5', '4_6', '4_7', '5', '5_6', '5_7', '6', '6_9', '7', '8', '8_10', '9']


In [37]:
def parse_ewg_score(raw_score) -> float:
    """
    None, "0"      → -1.0 (데이터 없음)
    [안전] ~       → 1.0
    [주의] ~       → 2.0
    [위험] ~       → 3.0
    hw_ewg 1~2     → 1.0 (안전)
    hw_ewg 3~6     → 2.0 (주의)
    hw_ewg 7~10    → 3.0 (위험)
    """
    if not raw_score:
        return -1.0

    raw_str = str(raw_score).strip()

    if raw_str == "0":
        return -1.0

    # coos_score: [안전]/[주의]/[위험] 파싱
    if "[안전]" in raw_str:
        return 1.0
    if "[주의]" in raw_str:
        return 2.0
    if "[위험]" in raw_str:
        return 3.0

    # hw_ewg: 숫자 범위 평균 구해서 등급 변환
    numbers = re.findall(r'\d+', raw_str)
    if not numbers:
        return -1.0

    avg = sum(float(n) for n in numbers) / len(numbers)

    if avg <= 2:
        return 1.0  # 안전
    elif avg <= 6:
        return 2.0  # 주의
    else:
        return 3.0  # 위험

# 테스트
test_cases = [
    ("0",              "missing"),
    ("[안전] 1 등급",   "안전"),
    ("[안전] 1-2 등급", "안전"),
    ("[주의] 3 등급",   "주의"),
    ("[주의] 1-6 등급", "주의"),
    ("[위험] 7-10 등급","위험"),
    ("1",              "안전"),
    ("1_2",            "안전"),
    ("3_6",            "주의"),
    ("7",              "위험"),
    ("8_10",           "위험"),
    (None,             "missing"),
]

for val, expected in test_cases:
    result = parse_ewg_score(val)
    print(f"{str(val):25} → {result}  ({expected})")

0                         → -1.0  (missing)
[안전] 1 등급                 → 1.0  (안전)
[안전] 1-2 등급               → 1.0  (안전)
[주의] 3 등급                 → 2.0  (주의)
[주의] 1-6 등급               → 2.0  (주의)
[위험] 7-10 등급              → 3.0  (위험)
1                         → 1.0  (안전)
1_2                       → 1.0  (안전)
3_6                       → 2.0  (주의)
7                         → 3.0  (위험)
8_10                      → 3.0  (위험)
None                      → -1.0  (missing)


### 청크 변환 함수

In [38]:
def row_to_chunks(row: dict) -> list[dict]:
    chunks = []
    ingredient    = str(row.get("ingredient_ko") or "")
    ingredient_en = str(row.get("ingredient_en") or "")

    base_meta = {
        "ingredient_ko":  ingredient,
        "ingredient_en":  ingredient_en,
        "coos_score":     row.get("coos_score"),
        "coos_score_num": parse_ewg_score(row.get("coos_score")),  # 🆕 숫자 변환
        "hw_ewg":         row.get("hw_ewg"),
        "hw_ewg_num":     parse_ewg_score(row.get("hw_ewg")),      # 🆕 숫자 변환
        "is_allergy":     row.get("hw_is_allergy"),
        "pc_rating":      row.get("pc_rating"),
    }

    # ── 청크 1: EWG 등급 ──────────────────────
    ewg_parts = []
    if row.get("coos_score"):
        ewg_parts.append(f"EWG 스코어: {row['coos_score']}")
    if row.get("coos_data_grade"):
        ewg_parts.append(f"데이터 등급: {row['coos_data_grade']}")
    if row.get("hw_ewg"):
        ewg_parts.append(f"화해 EWG: {row['hw_ewg']}")
    if row.get("hw_ewg_data_availability_text"):
        ewg_parts.append(f"EWG 데이터: {row['hw_ewg_data_availability_text']}")

    if ewg_parts:
        chunks.append({
            "page_content": f"[{ingredient}] " + " / ".join(ewg_parts),
            "metadata": {
                **base_meta,
                "chunk_type":   "ewg",
                "chunk_weight": CHUNK_WEIGHT["ewg"],
                "source":       "coos",
            }
        })

    # ── 청크 2: 성분 기본정보 ─────────────────
    basic_parts = []
    for col, label in [
        ("ingredient_en",      "INCI"),
        ("coos_function",      "기능"),
        ("coos_type",          "종류"),
        ("coos_country",       "국가"),
        ("coos_kr_restricted", "국내규제"),
        ("coos_eu_restricted", "유럽규제"),
        ("pc_effect",          "효과"),
        ("pc_category",        "분류"),
        ("hw_purpose",         "화장품목적"),
        ("hw_category",        "제품카테고리"),
    ]:
        val = row.get(col)
        if is_valid_value(val):  # 🆕 강화된 필터 적용
            basic_parts.append(f"{label}: {val}")

    if basic_parts:
        chunks.append({
            "page_content": f"[{ingredient}] " + " / ".join(basic_parts),
            "metadata": {
                **base_meta,
                "chunk_type":   "basic_info",
                "chunk_weight": CHUNK_WEIGHT["basic_info"],
                "source":       "coos",
            }
        })

    # ── 청크 3: 전문가 설명 ───────────────────
    expert_parts = []
    if row.get("pc_description"):
        expert_parts.append(f"PC 설명: {row['pc_description']}")
    if row.get("coos_ai_description"):
        expert_parts.append(f"AI 설명: {row['coos_ai_description']}")

    if expert_parts:
        chunks.append({
            "page_content": f"[{ingredient}] " + " | ".join(expert_parts),
            "metadata": {
                **base_meta,
                "chunk_type":   "expert",
                "chunk_weight": CHUNK_WEIGHT["expert"],
                "source":       "pc",
            }
        })

    # ── 청크 4: 리뷰·피부반응 ────────────────
    review_parts = []
    if row.get("hw_topics_positive"):
        review_parts.append(f"긍정 반응: {row['hw_topics_positive']}")
    if row.get("hw_topics_negative"):
        review_parts.append(f"부정 반응: {row['hw_topics_negative']}")
    if row.get("hw_primary_attr"):
        review_parts.append(f"주요속성: {row['hw_primary_attr']}")

    if review_parts:
        chunks.append({
            "page_content": f"[{ingredient}] " + " / ".join(review_parts),
            "metadata": {
                **base_meta,
                "chunk_type":   "review",
                "chunk_weight": CHUNK_WEIGHT["review"],
                "source":       "hw",
            }
        })

    return chunks

print("청크 변환 함수 정의 완료")

청크 변환 함수 정의 완료


### 전체 청크 변환 (중복 제거 적용)
`seen_ingredients` set을 추가해서 중복 제거.  
set은 **출석부** 같은 역할 — 이미 처리한 성분을 기록해둠.

- ewg / basic_info / expert → 성분별 1회만 생성
- review → 제품마다 내용이 다르므로 중복 제거 없이 전부 유지

In [39]:
all_chunks = []
seen_ingredients = set()  # 이미 처리한 성분 기록

for row in data:
    ingredient = row.get("ingredient_ko", "")

    if ingredient not in seen_ingredients:
        # 처음 보는 성분 → ewg/basic_info/expert 청크 생성
        seen_ingredients.add(ingredient)
        chunks = row_to_chunks(row)
        all_chunks.extend([c for c in chunks if c["metadata"]["chunk_type"] != "review"])

    # review는 제품별로 다르니까 중복 제거 없이 전부 추가
    chunks = row_to_chunks(row)
    review_chunks = [c for c in chunks if c["metadata"]["chunk_type"] == "review"]
    all_chunks.extend(review_chunks)

print(f"총 청크 수: {len(all_chunks)}")

type_counts = Counter(c["metadata"]["chunk_type"] for c in all_chunks)
print(f"청크 타입별 분포: {dict(type_counts)}")

총 청크 수: 121540
청크 타입별 분포: {'ewg': 16637, 'basic_info': 17144, 'expert': 16620, 'review': 71139}


### 제품 DB 저장

In [40]:
product_cols = [
    "ingredient_ko", "ingredient_en",
    "hw_product_id", "hw_product_name", "hw_brand_name",
    "hw_avg_ratings", "hw_review_count",
    "hw_price", "hw_consumer_price",
    "hw_primary_attr", "hw_category",
    "hw_is_allergy", "hw_limitation", "hw_forbidden",
]

df_product = pd.DataFrame(data)[product_cols]
df_product = df_product.dropna(subset=["hw_product_name"])

PRODUCT_DB_PATH = f"{BASE_PATH}/product_db.csv"
df_product.to_csv(PRODUCT_DB_PATH, index=False, encoding="utf-8-sig")

print(f"제품 DB 저장 완료: {PRODUCT_DB_PATH}")
print(f"제품 DB 행 수: {len(df_product)}")

제품 DB 저장 완료: /content/product_db.csv
제품 DB 행 수: 71139


### 청크 파일 저장

In [41]:
CHUNKS_PATH = f"{BASE_PATH}/ingredient_chunks.json"

with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

print(f"청크 저장 완료: {CHUNKS_PATH}")
print(f"총 청크 수: {len(all_chunks)}")

청크 저장 완료: /content/ingredient_chunks.json
총 청크 수: 121540
